### Notebook 03 – Exploratory Data Analysis (EDA)

##### Objective

understand how active mobility changes across Brisbane using the mastered dataset.

This notebook answers the following questions:

1. How many observations are available?
2. Which transport mode is most common?
3. Which locations are the busiest?
4. How does traffic change over time?
5. How do weekdays differ from weekends?
6. Are there seasonal trends?
7. Which locations have the highest activity?

### Dataset Overview

In [78]:
import pandas as pd
import numpy as np

import plotly.express as px

pd.set_option("display.max_columns", None)

In [9]:
df = pd.read_csv("../data/processed/master_dataset.csv")

In [10]:
df.head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013


In [11]:
df["Date"] = pd.to_datetime(df["Date"])

In [12]:
#basic info
print(f"Rows : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"Unique Sites : {df['Site_ID'].nunique()}")
print(f"Transport Modes : {df['Mode'].nunique()}")
print(f"Date Range : {df['Date'].min().date()} → {df['Date'].max().date()}")

Rows : 57,752
Columns : 15
Unique Sites : 39
Transport Modes : 3
Date Range : 2022-01-01 → 2025-12-31


##### Observation

The analytical dataset contains daily observations collected from automatic monitoring sites across Brisbane.

The dataset includes:

- Multiple transport modes
- Geographic information
- Temporal features
- Counter installation information


### Transport Mode Analysis

In [13]:
#Which transport mode contributes the highest overall traffic?
mode_volume = (
    df.groupby("Mode")["Count"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

In [14]:
fig = px.bar(
    mode_volume,
    x="Mode",
    y="Count",
    text="Count",
    color="Mode",
    title="Total Recorded Movements by Transport Mode"
)

fig.update_traces(texttemplate="%{text:,.0f}")

fig.show()

#### Daily traffic per counter for each traffic mode

In [29]:
counter_daily_avg = (
    df.groupby(["Mode", "Site_ID", "Site Name"])["Count"]
      .mean()
      .reset_index()
)

counter_daily_avg.head()

,Mode,Site_ID,Site Name,Count
0,Cyclist,A001,"Bicentennial Bikeway, Auchenflower",2563.479683
1,Cyclist,A002,"Bishop Street, Kelvin Grove",104.918987
2,Cyclist,A003,"Ekibin Park, Greenslopes",220.014286
3,Cyclist,A004E,"Eleanor Schonell Br Cyclists, St Lucia",778.701887
4,Cyclist,A005W,"Go Between Br Cyclists, South Brisbane",763.044554


In [30]:
mode_counter_avg = (
    counter_daily_avg.groupby("Mode")["Count"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
)

mode_counter_avg

,Mode,Count
0,Pedestrian,1332.351774
1,Cyclist,675.953914
2,Scooter,521.363405


In [31]:
fig = px.bar(
    mode_counter_avg,
    x="Mode",
    y="Count",
    color="Mode",
    text="Count",
    title="Average Daily Traffic per Monitoring Site by Transport Mode"
)

fig.update_traces(texttemplate="%{text:.0f}")

fig.update_layout(
    yaxis_title="Average Daily Movements",
    xaxis_title="Transport Mode",
    showlegend=False
)

fig.show()

##### Key Insight
Pedestrian traffic contributes the largest share of active mobility across Brisbane. This suggests that pedestrian infrastructure experiences the highest utilisation and should remain a priority for maintenance and future investment.

### Which locations are the busiest?

In [15]:
#Which monitoring sites experience the highest traffic volume?
top_sites = (
    df.groupby("Site Name")["Count"]
      .sum()
      .sort_values(ascending=False)
      .head(15)
      .reset_index()
)

In [16]:
fig = px.bar(
    top_sites,
    x="Count",
    y="Site Name",
    orientation="h",
    text="Count",
    title="Top 15 Busiest Monitoring Sites"
)

fig.update_layout(yaxis=dict(categoryorder="total ascending"))

fig.update_traces(texttemplate="%{text:,.0f}")

fig.show()

##### Among the automatic monitoring sites included in the dataset, the Maritime Museum counter recorded the highest total number of movements over the study period (2022–2025).

### How has active mobility changed over time in Brisbane?

In [17]:
daily_traffic = (
    df.groupby("Date")["Count"]
      .sum()
      .reset_index()
)

daily_traffic.head()

,Date,Count
0,2022-01-01,21876.0
1,2022-01-02,44869.0
2,2022-01-03,40165.0
3,2022-01-04,32782.0
4,2022-01-05,31279.0


In [18]:
fig = px.line(
    daily_traffic,
    x="Date",
    y="Count",
    title="Daily Active Mobility Across Brisbane",
    labels={
        "Count": "Total Daily Movements"
    }
)

fig.show()

In [19]:
monthly = (
    df.groupby(["Year", "Month", "Month_Name"])["Count"]
      .sum()
      .reset_index()
)

monthly = monthly.sort_values(["Year", "Month"])

In [20]:
fig = px.line(
    monthly,
    x="Month_Name",
    y="Count",
    color="Year",
    markers=True,
    title="Monthly Active Mobility by Year"
)

fig.show()

#### Which day of the week has been busisest?

In [21]:
dow = (
    df.groupby("Day_of_Week")["Count"]
      .mean()
      .reset_index()
)

In [22]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow["Day_of_Week"] = pd.Categorical(
    dow["Day_of_Week"],
    categories=day_order,
    ordered=True
)

dow = dow.sort_values("Day_of_Week")

In [23]:
fig = px.bar(
    dow,
    x="Day_of_Week",
    y="Count",
    text="Count",
    title="Average Daily Traffic by Day of Week"
)

fig.update_traces(texttemplate="%{text:,.0f}")

fig.show()

In [24]:
weekday = (
    df.groupby("Weekday_Weekend")["Count"]
      .mean()
      .reset_index()
)

In [25]:
fig = px.bar(
    weekday,
    x="Weekday_Weekend",
    y="Count",
    text="Count",
    color="Weekday_Weekend",
    title="Average Traffic: Weekdays vs Weekends"
)

fig.update_traces(texttemplate="%{text:,.0f}")

fig.show()

In [26]:
site_volume = (
    df.groupby(
        ["Site_ID", "Site Name", "Latitude", "Longitude"]
    )["Count"]
      .sum()
      .reset_index()
)

In [27]:
fig = px.scatter_map(
    site_volume,
    lat="Latitude",
    lon="Longitude",
    size="Count",
    color="Count",
    hover_name="Site Name",
    zoom=10,
    height=700,
    title="Active Mobility Monitoring Sites Across Brisbane"
)

fig.show()

#### Average traffic by season

In [36]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    else:
        return "Spring"

df["Season"] = df["Month"].apply(get_season)

In [37]:
season_avg = (
    df.groupby("Season")["Count"]
      .mean()
      .reindex([
          "Summer",
          "Autumn",
          "Winter",
          "Spring"
      ])
      .reset_index()
)

season_avg

,Season,Count
0,Summer,846.919268
1,Autumn,877.489053
2,Winter,861.993070
3,Spring,897.743099


In [38]:
fig = px.bar(
    season_avg,
    x="Season",
    y="Count",
    color="Season",
    text="Count",
    title="Average Daily Active Mobility by Season"
)

fig.update_traces(texttemplate="%{text:.0f}")

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Average Daily Recorded Movements",
    showlegend=False
)

fig.show()

#### Active mobility is highest during Spring and Autumn, suggesting that mild temperatures encourage walking and cycling. Traffic declines during Summer, possibly because of Brisbane's hotter weather.

### Top cyclist monitoring sites

In [40]:
cyclist_df = df[df["Mode"] == "Cyclist"]
top_cyclist = (
    cyclist_df.groupby("Site Name")["Count"]
              .sum()
              .sort_values(ascending=False)
              .head(15)
              .reset_index()
)

top_cyclist

,Site Name,Count
0,"Bicentennial Bikeway, Auchenflower",2586551.0
1,"Maritime Museum, South Brisbane",2250376.0
2,"Bicentennial Bikeway, Milton",1194276.0
3,"Stanley Street, South Brisbane",1126606.0
4,"Kurilpa Bridge Tank Street, Brisbane City",977325.0
5,"Kangaroo Point Bikeway, Kangaroo Point",932760.0
6,"Annerley Rd, South Brisbane",865354.0
7,"Lores Bonney Riverwalk, Hamilton",865106.0
8,"Jack Pesch Bridge, Indooroopilly",723204.0
9,"Riverwalk, Newfarm",678578.0


In [41]:
fig = px.bar(
    top_cyclist.sort_values("Count"),
    x="Count",
    y="Site Name",
    orientation="h",
    text="Count",
    title="Top 15 Cyclist Monitoring Sites"
)

fig.update_traces(texttemplate="%{text:,.0f}")

fig.update_layout(
    xaxis_title="Total Cyclist Movements",
    yaxis_title=""
)

fig.show()

#### Top pedestrian monitoring sites

In [42]:
pedestrian_df = df[df["Mode"] == "Pedestrian"]

top_pedestrian = (
    pedestrian_df.groupby("Site Name")["Count"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(15)
                 .reset_index()
)

In [43]:
fig = px.bar(
    top_pedestrian.sort_values("Count"),
    x="Count",
    y="Site Name",
    orientation="h",
    text="Count",
    title="Top 15 Pedestrian Monitoring Sites"
)

fig.update_traces(texttemplate="%{text:,.0f}")

fig.show()

#### Top scooter monitoring sites

In [46]:
scooter_df = df[df["Mode"] == "Scooter"]
top_scooter = (
    scooter_df.groupby("Site Name")["Count"]
              .sum()
              .sort_values(ascending=False)
              .head(15)
              .reset_index()
)

top_scooter

,Site Name,Count
0,"City Link Cycleway (Victoria Bridge), City",394990.0
1,"Kangaroo Point Green Bridge, Brisbane City",314597.0
2,City Link Cycleway (Elizabeth St - Edward to A...,266061.0
3,City Link Cycleway (Edward St - Charlotte to M...,203668.0
4,"Story Bridge Underpass, Kangaroo Point",168061.0


In [47]:
fig = px.bar(
    top_scooter.sort_values("Count"),
    x="Count",
    y="Site Name",
    orientation="h",
    text="Count",
    title="Top 15 Scooter Monitoring Sites"
)

fig.update_traces(texttemplate="%{text:,.0f}")

fig.update_layout(
    xaxis_title="Total Scooter Movements",
    yaxis_title=""
)

fig.show()

#### Site specialization - what is each monitoring site primarily used for?

In [48]:
site_mode = (
    df.groupby(["Site Name", "Mode"])["Count"]
      .sum()
      .reset_index()
)

site_mode.head()

,Site Name,Mode,Count
0,"Annerley Rd, South Brisbane",Cyclist,865354.0
1,"Bicentennial Bikeway, Auchenflower",Cyclist,2586551.0
2,"Bicentennial Bikeway, Auchenflower",Pedestrian,1538903.0
3,"Bicentennial Bikeway, Milton",Cyclist,1194276.0
4,"Bicentennial Bikeway, Milton",Pedestrian,699196.0


In [49]:
site_mode_pivot = (
    site_mode.pivot(
        index="Site Name",
        columns="Mode",
        values="Count"
    )
    .fillna(0)
)

In [55]:
site_percentage = (
    site_mode_pivot.div(
        site_mode_pivot.sum(axis=1),
        axis=0
    ) * 100
)

site_percentage.head(20)

Mode,Cyclist,Pedestrian,Scooter
Site Name,,,
"Annerley Rd, South Brisbane",100.000000,0.000000,0.000000
"Bicentennial Bikeway, Auchenflower",62.697366,37.302634,0.000000
"Bicentennial Bikeway, Milton",63.073338,36.926662,0.000000
"Bishop Street, Kelvin Grove",23.411347,76.588653,0.000000
"Botanic Gardens, City",23.278663,76.721337,0.000000
"Breakfast Creek / Yowoggera Bridge, Newstead",20.463789,79.536211,0.000000
"Bridge St, Wooloowin",77.229614,22.770386,0.000000
"City Link Cycleway (Edward St - Charlotte to Mary), City",59.582626,0.000000,40.417374
"City Link Cycleway (Elizabeth St - Creek to Edward), City",100.000000,0.000000,0.000000


In [52]:
top_sites = (
    site_mode.groupby("Site Name")["Count"]
             .sum()
             .sort_values(ascending=False)
             .head(15)
             .index
)

In [53]:
plot_df = site_percentage.loc[top_sites].reset_index()
plot_df = plot_df.melt(
    id_vars="Site Name",
    var_name="Mode",
    value_name="Percentage"
)

In [57]:
fig = px.bar(
    plot_df,
    x="Site Name",
    y="Percentage",
    color="Mode",
    title="Distribution of the 15 Busiest Monitoring Sites"
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_title="%age of Recorded Movements"
)

fig.show()

#### Monitoring sites are majorly dominated by cyclists and pedestrians and are likely to be associated with recreational areas, tourist destinations, or mixed-use public spaces.

### Fastest growing monitoring sites

In [58]:
site_year = (
    df.groupby(["Site_ID", "Site Name", "Year"])["Count"]
      .mean()
      .reset_index()
)

site_year.head()

,Site_ID,Site Name,Year,Count
0,A001,"Bicentennial Bikeway, Auchenflower",2022,2086.491758
1,A001,"Bicentennial Bikeway, Auchenflower",2023,2346.393151
2,A001,"Bicentennial Bikeway, Auchenflower",2024,2371.635394
3,A001,"Bicentennial Bikeway, Auchenflower",2025,2341.155844
4,A002,"Bishop Street, Kelvin Grove",2022,264.654147


In [59]:
growth_df = site_year.pivot(
    index=["Site_ID", "Site Name"],
    columns="Year",
    values="Count"
).reset_index()

growth_df.head()

Year,Site_ID,Site Name,2022,2023,2024,2025
0,A001,"Bicentennial Bikeway, Auchenflower",2086.491758,2346.393151,2371.635394,2341.155844
1,A002,"Bishop Street, Kelvin Grove",264.654147,127.532258,NaN,NaN
2,A003,"Ekibin Park, Greenslopes",316.403571,NaN,NaN,NaN
3,A004E,"Eleanor Schonell Br Cyclists, St Lucia",742.504110,735.129771,925.297619,NaN
4,A004N,"Eleanor Schonell Br Pedestrians, St Lucia",1687.279452,1749.632877,1107.314706,1094.086957


In [61]:
growth_complete = growth_df.dropna(subset=[2022, 2025]).copy()

growth_complete.shape

(23, 6)

In [62]:
growth_complete["Growth"] = (
    growth_complete[2025] - growth_complete[2022]
)

In [63]:
growth_complete["Growth_%"] = (
    (growth_complete["Growth"] / growth_complete[2022]) * 100
)

In [64]:
top_growth = (
    growth_complete
        .sort_values("Growth_%", ascending=False)
        .head(15)
)

top_growth[
    ["Site Name", 2022, 2025, "Growth", "Growth_%"]
]

Year,Site Name,2022,2025,Growth,Growth_%
14,"Riverwalk, Newfarm",707.282427,1272.538328,565.255901,79.919404
11,"Kedron Brook Bikeway, Mitchelton",175.815534,278.767025,102.951491,58.556539
22,"Lores Bonney Riverwalk, Hamilton",1118.697479,1377.926027,259.228548,23.172355
28,"Kurilpa Bridge Tank Street, Brisbane City",569.517906,671.010959,101.493053,17.820871
15,"Schulz Canal Bridge, Nundah",161.580000,186.241096,24.661096,15.262468
19,"Macquarie Street, St Lucia",695.882030,785.545643,89.663613,12.884887
0,"Bicentennial Bikeway, Auchenflower",2086.491758,2341.155844,254.664086,12.205372
26,"Maritime Museum, South Brisbane",3462.912935,3797.028409,334.115474,9.648394
21,"Gympie Rd (Marchant Park), Chermside",71.886686,75.175809,3.289123,4.575427
13,"North Brisbane Bikeway (Mann Park), Windsor",330.187050,344.824074,14.637024,4.432949


In [65]:
fig = px.bar(
    top_growth.sort_values("Growth_%"),
    x="Growth_%",
    y="Site Name",
    orientation="h",
    text="Growth_%",
    title="Top 15 Fastest Growing Monitoring Sites (2022–2025)"
)

fig.update_traces(
    texttemplate="%{text:.1f}%"
)

fig.update_layout(
    xaxis_title="Percentage Growth in Average Daily Traffic",
    yaxis_title=""
)

fig.show()

In [66]:
top_absolute = (
    growth_complete
        .sort_values("Growth", ascending=False)
        .head(15)
)

In [67]:
fig = px.bar(
    top_absolute.sort_values("Growth"),
    x="Growth",
    y="Site Name",
    orientation="h",
    text="Growth",
    title="Top 15 Monitoring Sites by Absolute Growth (2022–2025)"
)

fig.update_traces(texttemplate="%{text:.0f}")

fig.update_layout(
    xaxis_title="Increase in Average Daily Traffic",
    yaxis_title=""
)

fig.show()

In [68]:
growth_filtered = growth_complete[
    growth_complete[2022] >= 100
].copy()

In [69]:
top_percentage = (
    growth_filtered
        .sort_values("Growth_%", ascending=False)
        .head(15)
)

In [70]:
fig = px.bar(
    top_percentage.sort_values("Growth_%"),
    x="Growth_%",
    y="Site Name",
    orientation="h",
    text="Growth_%",
    title="Top 15 Monitoring Sites by Percentage Growth (2022–2025)"
)

fig.update_traces(
    texttemplate="%{text:.1f}%"
)

fig.update_layout(
    xaxis_title="Percentage Growth",
    yaxis_title=""
)

fig.show()

### Counter age vs Traffic

In [91]:
counter_age = (
    df.groupby(["Site_ID", "Site Name", "Start_Date"])["Count"]
      .mean()
      .reset_index()
)

counter_age.head()

,Site_ID,Site Name,Start_Date,Count
0,A001,"Bicentennial Bikeway, Auchenflower",2013,2299.584169
1,A002,"Bishop Street, Kelvin Grove",2015,252.526391
2,A003,"Ekibin Park, Greenslopes",2015,316.403571
3,A004E,"Eleanor Schonell Br Cyclists, St Lucia",2013,778.701887
4,A004N,"Eleanor Schonell Br Pedestrians, St Lucia",2013,1468.000000


In [94]:
latest_year = df["Year"].max()

counter_age["Counter_Age"] = (
    latest_year - counter_age["Start_Date"]
)

In [95]:
counter_age["Age_Group"] = pd.cut(
    counter_age["Counter_Age"],
    bins=[0, 3, 6, 9, 12, 15],
    labels=[
        "0–3 yrs",
        "4–6 yrs",
        "7–9 yrs",
        "10–12 yrs",
        "13–15 yrs"
    ]
)

In [96]:
fig = px.box(
    counter_age,
    x="Age_Group",
    y="Count",
    color="Age_Group",
    points="all",
    title="Traffic Distribution by Counter Age"
)

fig.update_layout(
    xaxis_title="Counter Age",
    yaxis_title="Average Daily Recorded Movements",
    showlegend=False
)

fig.show()

#### How has Brisbane expanded its automatic monitoring network over time?

In [97]:
installations = (
    df.groupby("Start_Date")
    .size()
    .reset_index(name="Counters_Installed")
)

In [98]:
installations = installations.sort_values("Start_Date")

In [99]:
fig = px.bar(
    installations,
    x="Start_Date",
    y="Counters_Installed",
    text="Counters_Installed",
    title="Expansion of Brisbane's Automatic Monitoring Network"
)

fig.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Installation Year",
    yaxis_title="Number of Monitoring Sites"
)

fig.show()

### Correlation Heatmap

In [111]:
mode_map = {
    "Cyclist": 0,
    "Pedestrian": 1,
    "Scooter": 2
}

df["Mode_Code"] = df["Mode"].map(mode_map)

In [112]:
day_map = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

df["Day_Code"] = df["Day_of_Week"].map(day_map)

In [113]:
week_map = {
    "Weekday": 0,
    "Weekend": 1
}

df["Weekend_Code"] = df["Weekday_Weekend"].map(week_map)

In [114]:
corr = df[
    [
        "Count",
        "Latitude",
        "Longitude",
        "Year",
        "Month",
        "Day_Code",
        "Weekend_Code",
        "Mode_Code"
    ]
].corr()


In [115]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Correlation Heatmap"
)

fig.show()

#### Monthly X Transport mode heatmap

In [116]:
heatmap_df = (
    df.groupby(["Month", "Mode"])["Count"]
      .mean()
      .reset_index()
)

In [117]:
heatmap_matrix = heatmap_df.pivot(
    index="Mode",
    columns="Month",
    values="Count"
)

In [118]:
fig = px.imshow(
    heatmap_matrix,
    text_auto=".0f",
    color_continuous_scale="Viridis",
    aspect="auto",
    title="Average Daily Traffic by Month and Transport Mode"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Transport Mode"
)

fig.show()

### Weekend percentage heatmap

In [120]:
weekend = (
    df.groupby(["Day_of_Week", "Mode"])["Count"]
      .mean()
      .reset_index()
)

In [122]:
weekend_matrix = weekend.pivot(
    index="Mode",
    columns="Day_of_Week",
    values="Count"
)

In [125]:
weekend_matrix.columns

Index(['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday',
       'Wednesday'],
      dtype='str', name='Day_of_Week')

In [126]:
weekend_matrix = weekend_matrix[
    [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ]
]

In [127]:
fig = px.imshow(
    weekend_matrix,
    text_auto=".0f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Average Daily Traffic by Day of Week"
)

fig.update_layout(
    xaxis_title="Day of Week",
    yaxis_title="Transport Mode"
)

fig.show()

#### Has Brisbane's transport mix changed over time?

In [129]:
year_mode = (
    df.groupby(["Year", "Mode"])["Count"]
      .sum()
      .reset_index()
)

year_mode

,Year,Mode,Count
0,2022,Cyclist,5157004.0
1,2022,Pedestrian,6308225.0
2,2022,Scooter,482151.0
3,2023,Cyclist,6498901.0
4,2023,Pedestrian,7940712.0
5,2023,Scooter,364010.0
6,2024,Cyclist,5237494.0
7,2024,Pedestrian,5930942.0
8,2024,Scooter,53617.0
9,2025,Cyclist,4380657.0


In [130]:
year_mode["Percentage"] = (
    year_mode["Count"]
    / year_mode.groupby("Year")["Count"].transform("sum")
    * 100
)

year_mode.head()

,Year,Mode,Count,Percentage
0,2022,Cyclist,5157004.0,43.164309
1,2022,Pedestrian,6308225.0,52.800070
2,2022,Scooter,482151.0,4.035621
3,2023,Cyclist,6498901.0,43.900746
4,2023,Pedestrian,7940712.0,53.640328


In [131]:
fig = px.bar(
    year_mode,
    x="Year",
    y="Percentage",
    color="Mode",
    text=year_mode["Percentage"].round(1),
    title="Year-wise Distribution of Transport Modes",
    labels={"Percentage": "Share of Total Traffic (%)"},
)

fig.update_layout(
    yaxis_title="Percentage of Total Recorded Traffic",
    xaxis_title="Year",
    yaxis=dict(range=[0,100])
)

fig.update_traces(texttemplate="%{text}%", textposition="inside")

fig.show()

## Executive Summary

This exploratory analysis examined four years (2022–2025) of active mobility data collected from Brisbane's automatic pedestrian, cyclist, and scooter monitoring network.

The objective was to understand how people move throughout Brisbane, identify the busiest corridors, investigate temporal mobility patterns, and uncover infrastructure insights that can support smarter transport planning.

Overall, the analysis indicates that pedestrian movement dominates Brisbane's active transport network, while cycling forms a significant proportion of recorded traffic. Scooter usage remains comparatively small but demonstrates increasing importance in selected locations. Temporal analysis reveals strong weekday-weekend differences, clear seasonal behaviour, and substantial variation between monitoring sites depending on surrounding land use and infrastructure.

These insights establish a strong analytical foundation for predictive modelling and smart mobility decision support.

## Key Findings

### 1. Pedestrians account for the largest share of recorded mobility

Across all monitoring sites between 2022 and 2025, pedestrians consistently represented the highest proportion of recorded movements, followed by cyclists, while scooters contributed only a small percentage of overall traffic.

---

### 2. Traffic is concentrated at a small number of locations

The Maritime Museum monitoring site recorded the highest overall movement during the study period, followed by Bicentennial Bikeway and Kangaroo Point Green Bridge. These corridors appear to function as Brisbane's primary active transport routes.

---

### 3. Different monitoring sites serve different transport purposes

Several monitoring sites are heavily cyclist-oriented (such as Bicentennial Bikeway), while others primarily serve pedestrians (such as South Bank and riverfront locations). This indicates that infrastructure usage strongly depends on surrounding urban land use.

---

### 4. Active mobility follows strong weekly patterns

Traffic generally peaks during weekdays, reflecting commuter travel, while weekend movement shifts toward recreational corridors and public spaces.

---

### 5. Seasonal conditions influence movement

Traffic levels are highest during Spring and Autumn when Brisbane experiences milder weather. Summer shows comparatively lower activity, likely due to higher temperatures.

---

### 6. Brisbane's transport mix remains relatively stable

Although total mobility changes throughout the year, pedestrians continue to dominate the transport mix. Cyclist proportions remain relatively stable, while scooters show gradual growth from 2022 onwards.

---

### 7. Some monitoring sites are experiencing rapid growth

Several counters exhibit consistent year-over-year increases in average daily traffic, suggesting expanding use of Brisbane's active transport infrastructure.

---

### 8. Older counters are not necessarily busier

Counter age alone does not explain traffic volume. Some recently installed counters already record among the highest daily movements, indicating that location is a much stronger determinant of traffic than installation age.

---

### 9. Spatial location strongly influences traffic intensity

Monitoring sites located along river corridors, bridges, CBD connections and major bikeways consistently record substantially higher movement than suburban locations.

---

### 10. This dataset provides a strong foundation for predictive analytics

The cleaned and integrated dataset contains rich temporal and spatial information that can support future machine learning models for traffic prediction, anomaly detection and infrastructure planning.

## Conclusion

This exploratory analysis successfully transformed raw automatic counter data into actionable mobility insights for Brisbane.

The project demonstrates an end-to-end data analytics workflow including:

- Data understanding
- Data cleaning
- Feature engineering
- Data integration
- Exploratory data analysis
- Spatial analytics
- Temporal analytics
- Infrastructure analysis

The resulting dataset and insights provide a valuable foundation for future work involving traffic forecasting, anomaly detection, smart city dashboards and evidence-based transport planning.